In [ ]:
import os
import json
import time
from typing import List, Dict, Tuple

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from decord import VideoReader, cpu

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


# =========================================================
# PATHS
# =========================================================
DATA_ROOT = "./MSVD"
BLIP_DIR = "./blip_msvd_finetuned_final"
OUTPUT_JSON = "./blip_msvd_finetuned_final/blip_retrieval_style_results.json"

NUM_FRAMES = 8
FRAME_SELECTION = "middle"   # "middle" or "first"
MAX_NEW_TOKENS = 30
VIDEO_BATCH_SIZE = 8


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sample_frame_indices(total_frames: int, num_frames: int) -> List[int]:
    if total_frames <= 0:
        raise ValueError("Video has no frames")
    return np.linspace(0, total_frames - 1, num_frames, dtype=int).tolist()


def load_video_frames(video_path: str, num_frames: int) -> List[Image.Image]:
    vr = VideoReader(video_path, ctx=cpu(0))
    idxs = sample_frame_indices(len(vr), num_frames)
    frames = vr.get_batch(idxs).asnumpy()
    return [Image.fromarray(frame).convert("RGB") for frame in frames]


def build_test_set(data_root: str) -> Tuple[List[Dict], List[Dict]]:
    test_file = os.path.join(data_root, "msvd_test.json")
    video_root = os.path.join(data_root, "raw_videos")

    if not os.path.exists(test_file):
        raise FileNotFoundError(f"Missing file: {test_file}")
    if not os.path.exists(video_root):
        raise FileNotFoundError(f"Missing folder: {video_root}")

    records = load_json(test_file)

    videos = {}
    queries = []

    for item in records:
        video_name = item["video"]
        video_id = item.get("video_id", video_name)
        video_path = os.path.join(video_root, video_name)

        if not os.path.exists(video_path):
            continue

        if video_id not in videos:
            videos[video_id] = {
                "video_id": video_id,
                "video_name": video_name,
                "video_path": video_path,
            }

        caps = item["caption"]
        if isinstance(caps, str):
            caps = [caps]

        for cap in caps:
            cap = " ".join(str(cap).strip().split()).lower()
            if cap:
                queries.append({
                    "video_id": video_id,
                    "caption": cap
                })

    return queries, list(videos.values())


def pick_frame(frames: List[Image.Image], mode: str = "middle") -> Image.Image:
    if len(frames) == 0:
        raise ValueError("No frames found")

    if mode == "first":
        return frames[0]
    return frames[len(frames) // 2]


@torch.no_grad()
def generate_video_captions(
    model,
    processor,
    videos,
    device,
    num_frames,
    batch_size,
    max_new_tokens,
    frame_selection
):
    generated_captions = []
    video_ids = []

    batch_images = []
    batch_ids = []

    for item in tqdm(videos, desc="Preparing videos"):
        frames = load_video_frames(item["video_path"], num_frames)
        chosen = pick_frame(frames, frame_selection)

        batch_images.append(chosen)
        batch_ids.append(item["video_id"])

        if len(batch_images) == batch_size:
            inputs = processor(images=batch_images, return_tensors="pt").to(device)
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=3
            )
            preds = processor.batch_decode(outputs, skip_special_tokens=True)

            for pred, vid in zip(preds, batch_ids):
                generated_captions.append(" ".join(pred.strip().split()).lower())
                video_ids.append(vid)

            batch_images, batch_ids = [], []

    if batch_images:
        inputs = processor(images=batch_images, return_tensors="pt").to(device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=3
        )
        preds = processor.batch_decode(outputs, skip_special_tokens=True)

        for pred, vid in zip(preds, batch_ids):
            generated_captions.append(" ".join(pred.strip().split()).lower())
            video_ids.append(vid)

    return generated_captions, video_ids


def compute_metrics(similarity: np.ndarray, gt_video_ids: List[str], video_ids: List[str]) -> Dict[str, float]:
    video_id_to_idx = {vid: i for i, vid in enumerate(video_ids)}
    gt_indices = np.array([video_id_to_idx[v] for v in gt_video_ids], dtype=np.int64)

    ranked = np.argsort(-similarity, axis=1)
    ranks = []

    for i in range(len(gt_indices)):
        rank = int(np.where(ranked[i] == gt_indices[i])[0][0]) + 1
        ranks.append(rank)

    ranks = np.array(ranks)

    return {
        "R@1": float(np.mean(ranks <= 1)),
        "R@5": float(np.mean(ranks <= 5)),
        "R@10": float(np.mean(ranks <= 10)),
        "MRR": float(np.mean(1.0 / ranks)),
        "MeanRank": float(np.mean(ranks)),
        "MedianRank": float(np.median(ranks)),
        "Top1Accuracy": float(np.mean(ranks == 1)),
        "Top5Accuracy": float(np.mean(ranks <= 5)),
    }


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    print("cwd:", os.getcwd())
    print("DATA_ROOT exists:", os.path.exists(DATA_ROOT))
    print("BLIP_DIR exists:", os.path.exists(BLIP_DIR))

    queries, videos = build_test_set(DATA_ROOT)
    print(f"Queries: {len(queries)} | Videos: {len(videos)}")

    processor = BlipProcessor.from_pretrained(BLIP_DIR)
    model = BlipForConditionalGeneration.from_pretrained(BLIP_DIR).to(device)
    model.eval()

    # Step 1: Generate one BLIP caption per video
    cap_start = time.time()
    generated_video_captions, video_ids = generate_video_captions(
        model=model,
        processor=processor,
        videos=videos,
        device=device,
        num_frames=NUM_FRAMES,
        batch_size=VIDEO_BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        frame_selection=FRAME_SELECTION,
    )
    cap_time = time.time() - cap_start

    print("\nSample generated captions:")
    for i in range(min(5, len(video_ids))):
        print(video_ids[i], "->", generated_video_captions[i])

    # Step 2: Retrieval-style ranking using generated captions
    query_texts = [q["caption"] for q in queries]
    gt_video_ids = [q["video_id"] for q in queries]

    retrieval_start = time.time()

    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
    all_texts = generated_video_captions + query_texts
    tfidf = vectorizer.fit_transform(all_texts)

    video_matrix = tfidf[:len(generated_video_captions)]
    query_matrix = tfidf[len(generated_video_captions):]

    similarity = linear_kernel(query_matrix, video_matrix)

    retrieval_time = time.time() - retrieval_start

    metrics = compute_metrics(similarity, gt_video_ids, video_ids)

    # Total time per query, including BLIP caption generation + retrieval ranking
    metrics["avg_query_latency_sec"] = float((cap_time + retrieval_time) / max(len(query_texts), 1))

    print("\nBLIP Retrieval-Style Metrics")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    results = {
        "metrics": metrics,
        "num_queries": len(query_texts),
        "num_videos": len(video_ids),
        "frame_selection": FRAME_SELECTION,
        "sample_generated_captions": [
            {
                "video_id": video_ids[i],
                "generated_caption": generated_video_captions[i]
            }
            for i in range(min(20, len(video_ids)))
        ]
    }

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"\nSaved to {OUTPUT_JSON}")


main()

Using device: cuda
cwd: /app
DATA_ROOT exists: True
BLIP_DIR exists: True
Queries: 27763 | Videos: 670


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 1520.63it/s]
[transformers] The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Preparing videos: 100%|██████████| 670/670 [01:09<00:00,  9.58it/s]



Sample generated captions:
fr9H1WLcF1A_256_261 -> a table tennis player is swinging on a rope
wFPmKChNrhU_3_11 -> a man galloping on his horse
o4OsYxsNGMI_77_82 -> a woman cuts up some vegetables
rw9h_574HxE_59_66 -> a monkey pulls the tail of a dog
jDFn-1lXJ98_71_80 -> a monkey pulls the tail of a dog

BLIP Retrieval-Style Metrics
R@1: 0.0260
R@5: 0.0541
R@10: 0.0746
MRR: 0.0476
MeanRank: 282.3797
MedianRank: 258.0000
Top1Accuracy: 0.0260
Top5Accuracy: 0.0541
avg_query_latency_sec: 0.0025

Saved to ./blip_msvd_finetuned_final/blip_retrieval_style_results.json


In [ ]:
import json
import pandas as pd

RESULT_PATH = "./blip_msvd_finetuned_final/blip_retrieval_style_results.json"

with open(RESULT_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

print("BLIP Retrieval-Style Results\n")

df = pd.DataFrame(list(results["metrics"].items()),columns=["Metric", "Value"])
df

BLIP Retrieval-Style Results



,Metric,Value
0,R@1,0.025970
1,R@5,0.054065
2,R@10,0.074560
3,MRR,0.047622
4,MeanRank,282.379678
5,MedianRank,258.000000
6,Top1Accuracy,0.025970
7,Top5Accuracy,0.054065
8,avg_query_latency_sec,0.002542


In [ ]:
import os
import json
import time
from typing import List, Dict, Tuple

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from decord import VideoReader, cpu

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


# =========================================================
# PATHS
# =========================================================
DATA_ROOT = "./MSVD"
BLIP_DIR = "./blip_msvd_finetuned_final"
OUTPUT_JSON = "./blip_msvd_finetuned_final/blip_retrieval_style_results.json"

NUM_FRAMES = 8
FRAME_SELECTION = "middle"   # "middle" or "first"
MAX_NEW_TOKENS = 30
VIDEO_BATCH_SIZE = 8


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sample_frame_indices(total_frames: int, num_frames: int) -> List[int]:
    if total_frames <= 0:
        raise ValueError("Video has no frames")
    return np.linspace(0, total_frames - 1, num_frames, dtype=int).tolist()


def load_video_frames(video_path: str, num_frames: int) -> List[Image.Image]:
    vr = VideoReader(video_path, ctx=cpu(0))
    idxs = sample_frame_indices(len(vr), num_frames)
    frames = vr.get_batch(idxs).asnumpy()
    return [Image.fromarray(frame).convert("RGB") for frame in frames]


def build_test_set(data_root: str) -> Tuple[List[Dict], List[Dict]]:
    test_file = os.path.join(data_root, "msvd_test.json")
    video_root = os.path.join(data_root, "raw_videos")

    if not os.path.exists(test_file):
        raise FileNotFoundError(f"Missing file: {test_file}")
    if not os.path.exists(video_root):
        raise FileNotFoundError(f"Missing folder: {video_root}")

    records = load_json(test_file)

    videos = {}
    queries = []

    for item in records:
        video_name = item["video"]
        video_id = item.get("video_id", video_name)
        video_path = os.path.join(video_root, video_name)

        if not os.path.exists(video_path):
            continue

        if video_id not in videos:
            videos[video_id] = {
                "video_id": video_id,
                "video_name": video_name,
                "video_path": video_path,
            }

        caps = item["caption"]
        if isinstance(caps, str):
            caps = [caps]

        for cap in caps:
            cap = " ".join(str(cap).strip().split()).lower()
            if cap:
                queries.append({
                    "video_id": video_id,
                    "caption": cap
                })

    return queries, list(videos.values())


def pick_frame(frames: List[Image.Image], mode: str = "middle") -> Image.Image:
    if len(frames) == 0:
        raise ValueError("No frames found")

    if mode == "first":
        return frames[0]
    return frames[len(frames) // 2]


@torch.no_grad()
def generate_video_captions(
    model,
    processor,
    videos,
    device,
    num_frames,
    batch_size,
    max_new_tokens,
    frame_selection
):
    generated_captions = []
    video_ids = []

    batch_images = []
    batch_ids = []

    for item in tqdm(videos, desc="Preparing videos"):
        frames = load_video_frames(item["video_path"], num_frames)
        chosen = pick_frame(frames, frame_selection)

        batch_images.append(chosen)
        batch_ids.append(item["video_id"])

        if len(batch_images) == batch_size:
            inputs = processor(images=batch_images, return_tensors="pt").to(device)
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=3
            )
            preds = processor.batch_decode(outputs, skip_special_tokens=True)

            for pred, vid in zip(preds, batch_ids):
                generated_captions.append(" ".join(pred.strip().split()).lower())
                video_ids.append(vid)

            batch_images, batch_ids = [], []

    if batch_images:
        inputs = processor(images=batch_images, return_tensors="pt").to(device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=3
        )
        preds = processor.batch_decode(outputs, skip_special_tokens=True)

        for pred, vid in zip(preds, batch_ids):
            generated_captions.append(" ".join(pred.strip().split()).lower())
            video_ids.append(vid)

    return generated_captions, video_ids


def compute_metrics(similarity: np.ndarray, gt_video_ids: List[str], video_ids: List[str]) -> Dict[str, float]:
    video_id_to_idx = {vid: i for i, vid in enumerate(video_ids)}
    gt_indices = np.array([video_id_to_idx[v] for v in gt_video_ids], dtype=np.int64)

    ranked = np.argsort(-similarity, axis=1)
    ranks = []

    for i in range(len(gt_indices)):
        rank = int(np.where(ranked[i] == gt_indices[i])[0][0]) + 1
        ranks.append(rank)

    ranks = np.array(ranks)

    return {
        "R@1": float(np.mean(ranks <= 1)),
        "R@5": float(np.mean(ranks <= 5)),
        "R@10": float(np.mean(ranks <= 10)),
        "MRR": float(np.mean(1.0 / ranks)),
        "MeanRank": float(np.mean(ranks)),
        "MedianRank": float(np.median(ranks)),
        "Top1Accuracy": float(np.mean(ranks == 1)),
        "Top5Accuracy": float(np.mean(ranks <= 5)),
    }


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    print("cwd:", os.getcwd())
    print("DATA_ROOT exists:", os.path.exists(DATA_ROOT))
    print("BLIP_DIR exists:", os.path.exists(BLIP_DIR))

    queries, videos = build_test_set(DATA_ROOT)
    print(f"Queries: {len(queries)} | Videos: {len(videos)}")

    processor = BlipProcessor.from_pretrained(BLIP_DIR)
    model = BlipForConditionalGeneration.from_pretrained(BLIP_DIR).to(device)
    model.eval()

    # Step 1: Generate one BLIP caption per video
    cap_start = time.time()
    generated_video_captions, video_ids = generate_video_captions(
        model=model,
        processor=processor,
        videos=videos,
        device=device,
        num_frames=NUM_FRAMES,
        batch_size=VIDEO_BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        frame_selection=FRAME_SELECTION,
    )
    cap_time = time.time() - cap_start

    print("\nSample generated captions:")
    for i in range(min(5, len(video_ids))):
        print(video_ids[i], "->", generated_video_captions[i])

    # Step 2: Retrieval-style ranking using generated captions
    query_texts = [q["caption"] for q in queries]
    gt_video_ids = [q["video_id"] for q in queries]

    retrieval_start = time.time()

    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
    all_texts = generated_video_captions + query_texts
    tfidf = vectorizer.fit_transform(all_texts)

    video_matrix = tfidf[:len(generated_video_captions)]
    query_matrix = tfidf[len(generated_video_captions):]

    similarity = linear_kernel(query_matrix, video_matrix)

    retrieval_time = time.time() - retrieval_start

    metrics = compute_metrics(similarity, gt_video_ids, video_ids)

    # Total time per query, including BLIP caption generation + retrieval ranking
    metrics["avg_query_latency_sec"] = float((cap_time + retrieval_time) / max(len(query_texts), 1))

    print("\nBLIP Retrieval-Style Metrics")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    results = {
        "metrics": metrics,
        "num_queries": len(query_texts),
        "num_videos": len(video_ids),
        "frame_selection": FRAME_SELECTION,
        "sample_generated_captions": [
            {
                "video_id": video_ids[i],
                "generated_caption": generated_video_captions[i]
            }
            for i in range(min(20, len(video_ids)))
        ]
    }

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"\nSaved to {OUTPUT_JSON}")


main()

Using device: cuda
cwd: /app
DATA_ROOT exists: True
BLIP_DIR exists: True
Queries: 27763 | Videos: 670


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 1489.31it/s]
[transformers] The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Preparing videos: 100%|██████████| 670/670 [01:10<00:00,  9.53it/s]



Sample generated captions:
fr9H1WLcF1A_256_261 -> a table tennis player is swinging on a rope
wFPmKChNrhU_3_11 -> a man galloping on his horse
o4OsYxsNGMI_77_82 -> a woman cuts up some vegetables
rw9h_574HxE_59_66 -> a monkey pulls the tail of a dog
jDFn-1lXJ98_71_80 -> a monkey pulls the tail of a dog

BLIP Retrieval-Style Metrics
R@1: 0.0260
R@5: 0.0541
R@10: 0.0746
MRR: 0.0476
MeanRank: 282.3797
MedianRank: 258.0000
Top1Accuracy: 0.0260
Top5Accuracy: 0.0541
avg_query_latency_sec: 0.0026

Saved to ./blip_msvd_finetuned_final/blip_retrieval_style_results.json


In [ ]:
import json
import pandas as pd

RESULT_PATH = "./blip_msvd_finetuned_final/blip_retrieval_style_results.json"

with open(RESULT_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

print("BLIP Retrieval-Style Results\n")

df = pd.DataFrame(list(results["metrics"].items()),columns=["Metric", "Value"])
df

BLIP Retrieval-Style Results



,Metric,Value
0,R@1,0.025970
1,R@5,0.054065
2,R@10,0.074560
3,MRR,0.047622
4,MeanRank,282.379678
5,MedianRank,258.000000
6,Top1Accuracy,0.025970
7,Top5Accuracy,0.054065
8,avg_query_latency_sec,0.002554
